# ONT ORFanage Spidroin Full-Length Screening

Annotate CDS features for ONT isoforms with ORFanage, translate proteins with `gffread`, and screen high-confidence full-length spidroins with protein-level HMMER terminal-domain models.

Intermediate files are written to `data/interim/{TASK_NAME}/`; final tables and FASTA files are written to `data/processed/{TASK_NAME}/`.

## Configuration

Import packages and define run-specific paths. `TASK_NAME` includes a timestamp suffix so each run is isolated.

In [1]:
from datetime import datetime

import polars as pl

from spider_silkome_module import (
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROJ_ROOT,
    RAW_DATA_DIR,
    run_cmd,
)

2026-05-24 18:17:55.568 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome


In [2]:
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
TASK_NAME = f"ont_orfanage_spidroin_{RUN_TIMESTAMP}"

isoform_dir = PROJ_ROOT / "workflow" / "ONT_RNA_align" / "results" / "isoforms"
genome_dir = RAW_DATA_DIR / "spider_genome"
hmm_dir = INTERIM_DATA_DIR / "spider_silkome_20251222" / "hmmbuild_output"

interim_dir = INTERIM_DATA_DIR / TASK_NAME
processed_dir = PROCESSED_DATA_DIR / TASK_NAME

THREADS = 70
EVALUE = 1e-10
MIN_HMM_COVERAGE = 0.90
TERMINAL_WINDOW = 300
FORCE = False

SMOKE_SPECIES = "Araneus_ventricosus"
RUN_SMOKE_TEST = False

interim_dir, processed_dir

(PosixPath('/home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260524_181830'),
 PosixPath('/home/gyk/project/spider_silkome/data/processed/ont_orfanage_spidroin_20260524_181830'))

## Step 1: Preflight

Check that required executables and input directories are available.

In [3]:
preflight_tsv = interim_dir / "preflight.tsv"

run_cmd(
    f"pixi run python -m spider_silkome_module.ont_orfanage_spidroin preflight \
        --isoform-dir {isoform_dir} \
        --genome-dir {genome_dir} \
        --hmm-dir {hmm_dir} \
        --output {preflight_tsv}",
    outputs=[preflight_tsv],
    force=FORCE,
)

pl.read_csv(preflight_tsv, separator="\t")

2026-05-24 18:18:52.811 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome


2026-05-24 18:18:53.059 | INFO     | __main__:preflight:847 - executable:orfanage: True /home/gyk/project/spider_silkome/.pixi/envs/default/bin/orfanage
2026-05-24 18:18:53.061 | INFO     | __main__:preflight:847 - executable:gffread: True /home/gyk/project/spider_silkome/.pixi/envs/default/bin/gffread
2026-05-24 18:18:53.061 | INFO     | __main__:preflight:847 - executable:hmmsearch: True /home/gyk/project/spider_silkome/.pixi/envs/default/bin/hmmsearch
2026-05-24 18:18:53.061 | INFO     | __main__:preflight:847 - isoform_dir_exists: True /home/gyk/project/spider_silkome/workflow/ONT_RNA_align/results/isoforms
2026-05-24 18:18:53.061 | INFO     | __main__:preflight:847 - genome_dir_exists: True /home/gyk/project/spider_silkome/data/raw/spider_genome
2026-05-24 18:18:53.061 | INFO     | __main__:preflight:847 - hmm_dir_exists: True /home/gyk/project/spider_silkome/data/interim/spider_silkome_20251222/hmmbuild_output
2026-05-24 18:18:53.061 | INFO     | __main__:preflight:847 - isoform_

check,status,value
str,bool,str
"""executable:orfanage""",true,"""/home/gyk/project/spider_silko…"
"""executable:gffread""",true,"""/home/gyk/project/spider_silko…"
"""executable:hmmsearch""",true,"""/home/gyk/project/spider_silko…"
"""isoform_dir_exists""",true,"""/home/gyk/project/spider_silko…"
"""genome_dir_exists""",true,"""/home/gyk/project/spider_silko…"
"""hmm_dir_exists""",true,"""/home/gyk/project/spider_silko…"
"""isoform_gtf_count""",true,"""10"""
"""hmm_count""",true,"""29"""


## Step 2: Discover Species

Build a manifest that pairs ONT isoform GTF files with matching genome FASTA and template GFF annotations.

In [4]:
manifest_tsv = interim_dir / "species_manifest.tsv"

run_cmd(
    f"pixi run python -m spider_silkome_module.ont_orfanage_spidroin discover \
        --isoform-dir {isoform_dir} \
        --genome-dir {genome_dir} \
        --output {manifest_tsv}",
    outputs=[manifest_tsv],
    force=FORCE,
)

manifest = pl.read_csv(manifest_tsv, separator="\t")
manifest

2026-05-24 18:19:01.668 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome


2026-05-24 18:19:01.937 | SUCCESS  | __main__:discover:863 - Wrote /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260524_181830/species_manifest.tsv with 10/10 matched species


species,isoform_gtf,genome_fasta,template_gff,status,reason
str,str,str,str,str,str
"""Araneus_ventricosus""","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""matched""",null
"""Evarcha_sp""","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""matched""",null
"""Heteropoda_venatoria""","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""matched""",null
"""Hippasa_lycosina""","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""matched""",null
"""Pandercetes_sp""","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""matched""",null
"""Pardosa_pseudoannulata""","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""matched""",null
"""Pholcus_sp""","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""matched""",null
"""Scorpiops_zhui""","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""matched""",null
"""Songthela_sp""","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""matched""",null


## Optional Smoke Test

Run the complete workflow for one species before starting the full batch by setting `RUN_SMOKE_TEST = True` in the configuration cell.

In [ ]:
if RUN_SMOKE_TEST:
    smoke_interim_dir = interim_dir / "smoke_test"
    smoke_processed_dir = processed_dir / "smoke_test"
    smoke_target_tsv = smoke_processed_dir / "target_classification.tsv"

    run_cmd(
        f"pixi run python -m spider_silkome_module.ont_orfanage_spidroin run \
            --isoform-dir {isoform_dir} \
            --genome-dir {genome_dir} \
            --hmm-dir {hmm_dir} \
            --interim-dir {smoke_interim_dir} \
            --processed-dir {smoke_processed_dir} \
            --species {SMOKE_SPECIES} \
            --threads {THREADS} \
            --evalue {EVALUE} \
            --min-hmm-coverage {MIN_HMM_COVERAGE} \
            --terminal-window {TERMINAL_WINDOW}",
        outputs=[smoke_target_tsv],
        force=FORCE,
    )

    display(pl.read_csv(smoke_processed_dir / "orfanage_protein_qc.tsv", separator="\t"))
    display(pl.read_csv(smoke_target_tsv, separator="\t").head())

## Step 3: Run Full Workflow

Run ORFanage, translate ORFanage CDS, search HMMER terminal-domain profiles, and write final processed outputs.

In [6]:
target_classification_tsv = processed_dir / "target_classification.tsv"

run_cmd(
    f"pixi run python -m spider_silkome_module.ont_orfanage_spidroin run \
        --isoform-dir {isoform_dir} \
        --genome-dir {genome_dir} \
        --hmm-dir {hmm_dir} \
        --interim-dir {interim_dir} \
        --processed-dir {processed_dir} \
        --threads {THREADS} \
        --evalue {EVALUE} \
        --min-hmm-coverage {MIN_HMM_COVERAGE} \
        --terminal-window {TERMINAL_WINDOW}",
    outputs=[target_classification_tsv],
    force=True,
)

target_classification_tsv

2026-05-25 10:05:41.069 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome
Species:  10%|█         | 1/10 [00:04<00:36,  4.11s/it]

2026-05-25 10:05:45.371 | INFO     | __main__:run_command:218 - Skipping existing output: /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260524_181830/orfanage/Araneus_ventricosus.orfanage.gtf
2026-05-25 10:05:45.372 | INFO     | __main__:run_command:218 - Skipping existing output: /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260524_181830/proteins/Araneus_ventricosus.orfanage.proteins.faa
2026-05-25 10:05:45.449 | INFO     | __main__:run:1020 - Araneus_ventricosus: 17995 translated proteins
2026-05-25 10:05:45.449 | INFO     | __main__:run_command:218 - Skipping existing output: /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260524_181830/hmmer/Araneus_ventricosus.domtblout


Species:  10%|█         | 1/10 [00:08<00:36,  4.11s/it]loading reference genome
loading reference transcriptomes


2026-05-25 10:05:50.290 | INFO     | __main__:run:992 - Evarcha_sp: repaired 161 transcript rows and swapped 0 other malformed feature rows in template GFF
2026-05-25 10:05:50.291 | INFO     | __main__:run_command:222 - orfanage --reference /home/gyk/project/spider_silkome/data/raw/spider_genome/013.Evarcha_sp/Evarcha_sp.fa --query /home/gyk/project/spider_silkome/workflow/ONT_RNA_align/results/isoforms/Evarcha_sp.isoforms.gtf --output /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260524_181830/orfanage/Evarcha_sp.orfanage.gtf --stats /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260524_181830/orfanage/Evarcha_sp.stats.tsv --threads 70 --use_id --cleanq --cleant --rescue /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260524_181830/template_gff_cleaned/Evarcha_sp.cleaned.gff


sorting reference transcriptome
start removing duplicates
loading query transcriptome
bundling transcriptome
starting main evaluation
Done
Species:  10%|█         | 1/10 [00:58<00:36,  4.11s/it]

...

Species:  90%|█████████ | 9/10 [06:14<00:37, 37.85s/it]

2026-05-25 10:11:55.338 | INFO     | __main__:run:1020 - Trichonephila_clavata: 17914 translated proteins
2026-05-25 10:11:55.338 | INFO     | __main__:run_command:222 - hmmsearch --cpu 70 --noali -E 1e-10 --domE 1e-10 --incE 1e-10 --incdomE 1e-10 -o /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260524_181830/hmmer/Trichonephila_clavata.hmmsearch.txt --tblout /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260524_181830/hmmer/Trichonephila_clavata.tblout --domtblout /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260524_181830/hmmer/Trichonephila_clavata.domtblout /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260524_181830/hmmer/spidroin_terminals.all.hmm /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260524_181830/proteins/Trichonephila_clavata.orfanage.proteins.normalized.faa


Species: 100%|██████████| 10/10 [06:17<00:00, 37.71s/it]


2026-05-25 10:12:08.977 | SUCCESS  | __main__:run_classification:876 - Classified 22 targets: 1 full-length, 21 partial/non-full; wrote 1 FASTA records.


PosixPath('/home/gyk/project/spider_silkome/data/processed/ont_orfanage_spidroin_20260524_181830/target_classification.tsv')

## Results

Inspect final full-length candidates, pivot counts, and validation checks.

In [7]:
target_classification = pl.read_csv(processed_dir / "target_classification.tsv", separator="\t")
full_length = pl.read_csv(processed_dir / "full_length_spidroins.tsv", separator="\t")
partial = pl.read_csv(processed_dir / "partial_spidroins.tsv", separator="\t")
pivot = pl.read_csv(processed_dir / "spidroin_pivot_table.tsv", separator="\t")
validation = pl.read_csv(processed_dir / "validation_summary.tsv", separator="\t")

print(f"target classifications: {target_classification.shape}")
print(f"full-length candidates: {full_length.shape}")
print(f"partial/non-full candidates: {partial.shape}")

display(validation)
display(pivot)
full_length.head()

target classifications: (22, 38)
full-length candidates: (1, 38)
partial/non-full candidates: (21, 38)


check,status,value
str,bool,str
"""full_length_fasta_count_matche…",true,"""1/1"""
"""full_length_fasta_duplicate_id…",true,"""0"""
"""full_length_same_type_ntd_ctd""",true,"""0"""


species,MaSp
str,i64
"""Araneus_ventricosus""",0
"""Evarcha_sp""",0
"""Heteropoda_venatoria""",0
"""Hippasa_lycosina""",1
"""Pandercetes_sp""",0
"""Pardosa_pseudoannulata""",0
"""Pholcus_sp""",0
"""Scorpiops_zhui""",0
"""Songthela_sp""",0


species,target_id,tlen,classification,spidroin_type,ntd_profile,ctd_profile,ntd_spidroin_type,ctd_spidroin_type,ntd_end_ok,ctd_end_ok,ntd_c_evalue,ntd_dom_score,ntd_env_from,ntd_env_to,ntd_hmm_from,ntd_hmm_to,ntd_hmm_coverage,ntd_env_coverage,ctd_c_evalue,ctd_dom_score,ctd_env_from,ctd_env_to,ctd_hmm_from,ctd_hmm_to,ctd_hmm_coverage,ctd_env_coverage,num_ntd_hits,num_ctd_hits,original_transcript_id,gene_id,scaffold,strand,start,end,orfanage_status,orfanage_template,summed_dom_score
str,str,i64,str,str,str,str,str,str,bool,bool,f64,f64,i64,i64,i64,i64,f64,f64,f64,f64,i64,i64,i64,i64,f64,f64,i64,i64,str,str,str,str,i64,i64,i64,str,f64
"""Hippasa_lycosina""","""Hippasa_lycosina|g5993.t1""",1480,"""full_length""","""MaSp""","""MaSp_NTD""","""MaSp_CTD""","""MaSp""","""MaSp""",true,true,1.1000e-62,200.0,1,1424,1,252,0.913043,5.15942,9.9000e-51,160.5,1248,1480,5,233,0.982833,1.0,4,2,"""g5993.t1""","""g5993""","""Chr04""","""+""",139260392,139264834,1,"""g5993.t1""",360.5


In [8]:
processed_outputs = {
    "target_classification": processed_dir / "target_classification.tsv",
    "full_length_spidroins": processed_dir / "full_length_spidroins.tsv",
    "partial_spidroins": processed_dir / "partial_spidroins.tsv",
    "spidroin_pivot_table": processed_dir / "spidroin_pivot_table.tsv",
    "full_length_fasta": processed_dir / "full_length_spidroins.fasta",
    "by_type_dir": processed_dir / "by_type",
}

processed_outputs

{'target_classification': PosixPath('/home/gyk/project/spider_silkome/data/processed/ont_orfanage_spidroin_20260524_181830/target_classification.tsv'),
 'full_length_spidroins': PosixPath('/home/gyk/project/spider_silkome/data/processed/ont_orfanage_spidroin_20260524_181830/full_length_spidroins.tsv'),
 'partial_spidroins': PosixPath('/home/gyk/project/spider_silkome/data/processed/ont_orfanage_spidroin_20260524_181830/partial_spidroins.tsv'),
 'spidroin_pivot_table': PosixPath('/home/gyk/project/spider_silkome/data/processed/ont_orfanage_spidroin_20260524_181830/spidroin_pivot_table.tsv'),
 'full_length_fasta': PosixPath('/home/gyk/project/spider_silkome/data/processed/ont_orfanage_spidroin_20260524_181830/full_length_spidroins.fasta'),
 'by_type_dir': PosixPath('/home/gyk/project/spider_silkome/data/processed/ont_orfanage_spidroin_20260524_181830/by_type')}